# 01 FTH

            First notebook in the workflow. It loads the raw holograms, centers them,
            optionally applies the Ewald projection, defines an ROI, creates a simple
            radially symmetric Butterworth smooth beamstop mask, reconstructs the FTH
            image, and saves the nested `data` dictionary to HDF5.

            Output: `processed/Logs/data_recon_ImId_<im_id>_<user>.hdf5`.

In [1]:
# Configure Qt before importing pyplot or any other GUI-aware library.
import os
import sys
from os.path import join

BASEFOLDER = os.path.abspath(os.getcwd())
LIBRARY_FOLDER = join(BASEFOLDER, "library")
if LIBRARY_FOLDER not in sys.path:
    sys.path.insert(0, LIBRARY_FOLDER)

from notebook_setup import configure_matplotlib_qt
MATPLOTLIB_BACKEND = configure_matplotlib_qt()
print("Matplotlib backend:", MATPLOTLIB_BACKEND)

import os, sys
from os.path import join
from getpass import getuser
from importlib import reload

import h5py
import numpy as np
import matplotlib.pyplot as plt
import skimage.morphology



def find_basefolder(start=None):
    return os.path.abspath(start or os.getcwd())


BASEFOLDER = find_basefolder()
sys.path.append(join(BASEFOLDER, "library"))
print("Base folder:", BASEFOLDER)
import fthcore as fth
import helper_functions as helper
import interactive
from interactive import cimshow
import mask_lib
import reconstruct_rb as rec
import PETRA_MaxP04_loading as loading
import fth_phase_workflow as wf

wf = reload(wf)  # Refresh helpers when rerunning in an existing kernel.
from data_loading import SextantsNexusLoader, image_ids, load_average
from image_preprocessing import fit_dark_frame, fit_horizontal_band, load_detector_masks

try:
    import cupy as cp
    import cupyx as cpx
    import CCI_core_cupy as cci
    import Phase_Retrieval as PhR

    GPU = True
    print("GPU available")
except Exception:
    import CCI_core as cci

    PhR = None
    GPU = False
    print("GPU unavailable")
try:
    %load_ext jupyter_black
except Exception:
    pass
from pipeline_io import acquisition_ids as image_ids, first_id, result_path, stitched_path, stitched_ids, load_stitched

from pipeline_io import stitched_setup_metadata


Matplotlib backend: qt5 (pyqt5)
Base folder: /home/experiences/sextants/com-sextants/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS-2609_SOLEIL_SEXTANTS
GPU available


## Folders and user

In [2]:
BASEFOLDER = find_basefolder()
RAW_FOLDER = "/nfs/ruche/sextants-soleil/com-sextants/COMET_20261110_Battistelli_laser_RF/"
RAW_DATA_KIND = "sextants_nexus"
DATAFOLDER = RAW_FOLDER
USER = "rb"

folder_general = helper.create_folder(join(BASEFOLDER, "processed"))
folder_logs = helper.create_folder(join(folder_general, "Logs"))
print("Raw folder:", RAW_FOLDER)
print("Output folder:", folder_general)

Raw folder: /nfs/ruche/sextants-soleil/com-sextants/COMET_20261110_Battistelli_laser_RF/
Output folder: /home/experiences/sextants/com-sextants/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS-2609_SOLEIL_SEXTANTS/processed


In [3]:
# ls /nfs/ruche/sextants-soleil/com-sextants/COMET_20261110_Battistelli_laser_RF/

## Define raw images

In [526]:
# Current acquisition parameters from the newer working notebook.
# A single ID or a list is accepted; lists are averaged as floating point.

PLUS_IMAGE_IDS = 1564#list(np.arange(1144, 1164))
MINUS_IMAGE_IDS = 1565#list(np.arange(1164, 1184))

PLUS_IMAGE_IDS = 1598
MINUS_IMAGE_IDS = 1599


PLUS_IMAGE_IDS = 1866
MINUS_IMAGE_IDS = 1867
PLUS_DARK_IDS = 1837 #1790 after moving bs closer
MINUS_DARK_IDS = 1837 #1790


#PLUS_IMAGE_IDS = list(np.arange(1144, 1164))
#MINUS_IMAGE_IDS = list(np.arange(1164, 1184))
#PLUS_DARK_IDS = 904
#MINUS_DARK_IDS = 904

# The fixed detector mask is always processed/mask_pixels/mask_detector.png.
# These IDs select mask_beamstop_<ID>.png in the same folder.
PLUS_BEAMSTOP_MASK_ID = 1795#1795- after moving camera closer
MINUS_BEAMSTOP_MASK_ID = 1795#1795

# Choose "raw" or "stitched". Explicit legacy NPZ paths are also accepted.
INPUT_KIND = "raw"
PLUS_STITCHED_FILE = None
MINUS_STITCHED_FILE = None
if INPUT_KIND == "stitched":
    PLUS_STITCHED_FILE = PLUS_STITCHED_FILE or stitched_path(BASEFOLDER, PLUS_IMAGE_IDS, "+", USER)
    MINUS_STITCHED_FILE = MINUS_STITCHED_FILE or stitched_path(BASEFOLDER, PLUS_IMAGE_IDS, "-", USER)
    # The saved stitching reference IDs determine identity, including averages.
    PLUS_IMAGE_IDS = stitched_ids(BASEFOLDER, PLUS_STITCHED_FILE)
    MINUS_IMAGE_IDS = stitched_ids(BASEFOLDER, MINUS_STITCHED_FILE)
elif INPUT_KIND != "raw":
    raise ValueError("INPUT_KIND must be raw or stitched")
PLUS_IMAGE_IDS = image_ids(PLUS_IMAGE_IDS)
MINUS_IMAGE_IDS = image_ids(MINUS_IMAGE_IDS)

POSITIVE_LABEL = "+"
REFERENCE_LABEL = "-"
CROP = None
PROJECT_EWALD_SPHERE = False
EWALD_METHOD = "cubic"
HERALDO = False

MASK_FOLDER = join(BASEFOLDER, "processed", "mask_pixels")
MASK_DETECTOR_FILE = join(MASK_FOLDER, "mask_detector.png")
PLUS_BEAMSTOP_MASK_FILE = join(MASK_FOLDER, f"mask_beamstop_{PLUS_BEAMSTOP_MASK_ID}.png")
MINUS_BEAMSTOP_MASK_FILE = join(MASK_FOLDER, f"mask_beamstop_{MINUS_BEAMSTOP_MASK_ID}.png")

print("Fixed detector mask:", MASK_DETECTOR_FILE)
print("Plus beamstop mask:", PLUS_BEAMSTOP_MASK_FILE)
print("Minus beamstop mask:", MINUS_BEAMSTOP_MASK_FILE)

# This nested object is generated only for the established HDF5 format.
hologram_inputs = {
    "+": {
        "id": PLUS_IMAGE_IDS,
        "dark_id": PLUS_DARK_IDS,
        "beamstop_mask_id": PLUS_BEAMSTOP_MASK_ID,
        "stitched_file": PLUS_STITCHED_FILE,
    },
    "-": {
        "id": MINUS_IMAGE_IDS,
        "dark_id": MINUS_DARK_IDS,
        "beamstop_mask_id": MINUS_BEAMSTOP_MASK_ID,
        "stitched_file": MINUS_STITCHED_FILE,
    },
}
positive_label = POSITIVE_LABEL
reference_label = REFERENCE_LABEL
crop = CROP
project_ewalds_sphere = PROJECT_EWALD_SPHERE
ewald_method = EWALD_METHOD
heraldo = HERALDO


Fixed detector mask: /home/experiences/sextants/com-sextants/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS-2609_SOLEIL_SEXTANTS/processed/mask_pixels/mask_detector.png
Plus beamstop mask: /home/experiences/sextants/com-sextants/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS-2609_SOLEIL_SEXTANTS/processed/mask_pixels/mask_beamstop_1795.png
Minus beamstop mask: /home/experiences/sextants/com-sextants/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS-2609_SOLEIL_SEXTANTS/processed/mask_pixels/mask_beamstop_1795.png


## Derived paths and experimental setup

In [527]:
# Everything in this cell is derived from the folders and raw-image cells above.
raw_loader = (
    SextantsNexusLoader(RAW_FOLDER) if RAW_DATA_KIND == "sextants_nexus" else None
)
if raw_loader is None:
    raise ValueError(
        "Automatic experimental metadata currently requires RAW_DATA_KIND='sextants_nexus'"
    )

im_ids = image_ids(hologram_inputs[positive_label]["id"])

im_id = first_id(im_ids)

# New stitched files carry geometry, so no raw detector read is needed.
setup_metadata = (stitched_setup_metadata(BASEFOLDER, PLUS_STITCHED_FILE)
                  if PLUS_STITCHED_FILE is not None else {})
if "ccd_dist_m" not in setup_metadata or "energy_eV" not in setup_metadata:
    # Backward compatibility for older NPZ files, and the direct raw workflow.
    setup_metadata = {**dict(raw_loader.load(im_id).metadata), **setup_metadata}
if "ccd_dist_m" not in setup_metadata or "energy_eV" not in setup_metadata:
    raise KeyError(f"Missing energy or detector distance for positive image {im_id}")
ccd_dist_source = setup_metadata.get("ccd_dist_path", "stitched NPZ metadata")
ccd_dist_m = float(setup_metadata["ccd_dist_m"])
energy = float(setup_metadata["energy_eV"])
experimental_setup = {
    "ccd_dist": ccd_dist_m,
    "ccd_dist_source": ccd_dist_source,
    "ccd_dist_image_id": im_id,
    "px_size": 11.0e-6,
    "binning": 1,
    "oversaturation": 60e3,
    "energy": energy,
    "lambda": helper.photon_energy_wavelength(energy, input_unit="eV"),
}
mnemonics = loading.mnemonics
DATA_H5 = result_path(BASEFOLDER, im_id, USER)

data = {
    "workflow": "FTH_phase_retrieval_4_notebook_sequence",
    "user": USER,
    "data_file": DATA_H5,
    "experimental_setup": experimental_setup,
    "holo": hologram_inputs,
    "hologram_labels": list(hologram_inputs.keys()),
    "positive_label": positive_label,
    "reference_label": reference_label,
    "heraldo": heraldo,
    "crop": crop,
}

print("Positive images:", im_ids)
print("Output HDF5:", DATA_H5)
print(
    "CCD distance:",
    experimental_setup["ccd_dist"],
    "m from",
    experimental_setup["ccd_dist_source"],
)
print("Pixel size:", experimental_setup["px_size"], "m")
print("Energy:", experimental_setup["energy"], "eV")

Positive images: [1868]
Output HDF5: /home/experiences/sextants/com-sextants/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS-2609_SOLEIL_SEXTANTS/processed/Logs/data_recon_ImId_1868_rb.hdf5
CCD distance: 0.250002 m from scan_1868/scan_data/data_03
Pixel size: 1.1e-05 m
Energy: 784.2566942831609 eV


## Load raw data

In [528]:
from pipeline_io import acquisition_ids
# Dark normalization is always fitted in this corner and excludes detector defects.
DARK_FIT_ROWS = slice(0, 600)
DARK_FIT_COLUMNS = slice(0, 200)
DARK_FIT_PERCENTILE = 100
DARK_FIT_STRIDE = 1


def load_raw_or_stitched(image_ids, dark_ids, beamstop_mask_id, stitched_file, label):
    if stitched_file is not None:
        return load_stitched(BASEFOLDER, stitched_file)

    frame = load_average(raw_loader, acquisition_ids(image_ids))
    image = np.asarray(frame.image, dtype=float)
    mask_detector, mask_beamstop, mask_pixel = load_detector_masks(
        MASK_FOLDER, beamstop_mask_id, image.shape, detector_file=MASK_DETECTOR_FILE
    )

    # Fit average_image = scale * average_dark + offset in the selected corner.
    dark = np.asarray(load_average(raw_loader, acquisition_ids(dark_ids)).image, dtype=float)
    image, dark_scale, dark_offset, dark_values, fitted_pixels = fit_dark_frame(
        image,
        dark,
        DARK_FIT_ROWS,
        DARK_FIT_COLUMNS,
        percentile=DARK_FIT_PERCENTILE,
        stride=DARK_FIT_STRIDE,
        mask=mask_detector,
    )
    image_values = frame.image[DARK_FIT_ROWS, DARK_FIT_COLUMNS][
        ::DARK_FIT_STRIDE, ::DARK_FIT_STRIDE
    ].ravel()
    fig, axis = plt.subplots(figsize=(6, 5))
    axis.scatter(dark_values[fitted_pixels], image_values[fitted_pixels], s=3, alpha=0.15)
    fit_x = np.linspace(dark_values[fitted_pixels].min(), dark_values[fitted_pixels].max(), 200)
    axis.plot(fit_x, dark_scale * fit_x + dark_offset, color="red", linewidth=2)
    axis.set_title(f"{label}: fitted dark normalization")
    axis.set_xlabel("Dark intensity")
    axis.set_ylabel("Image intensity")
    axis.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
    cimshow(dark)
    print(f"{label}: dark scale={dark_scale:.6g}, offset={dark_offset:.6g}")
    return image, mask_detector, mask_beamstop, mask_pixel, dict(frame.metadata), frame.exposure, str(frame.source), (dark_scale, dark_offset)
data["dark_fit_recipe"] = {
    "rows": [DARK_FIT_ROWS.start, DARK_FIT_ROWS.stop],
    "columns": [DARK_FIT_COLUMNS.start, DARK_FIT_COLUMNS.stop],
    "percentile": DARK_FIT_PERCENTILE, "stride": DARK_FIT_STRIDE,
}


In [529]:
plus_loaded = load_raw_or_stitched(
    PLUS_IMAGE_IDS, PLUS_DARK_IDS, PLUS_BEAMSTOP_MASK_ID, PLUS_STITCHED_FILE, "+"
)
minus_loaded = load_raw_or_stitched(
    MINUS_IMAGE_IDS, MINUS_DARK_IDS, MINUS_BEAMSTOP_MASK_ID, MINUS_STITCHED_FILE, "-"
)

# Simple NumPy names are available immediately after loading.
pos_raw = np.asarray(plus_loaded[0], dtype=float)
neg_raw = np.asarray(minus_loaded[0], dtype=float)
mask_detector_pos = np.asarray(plus_loaded[1], dtype=np.uint8)
mask_detector_neg = np.asarray(minus_loaded[1], dtype=np.uint8)
mask_beamstop_pos = np.asarray(plus_loaded[2], dtype=np.uint8)
mask_beamstop_neg = np.asarray(minus_loaded[2], dtype=np.uint8)
mask_pixel_pos = np.asarray(plus_loaded[3], dtype=np.uint8)
mask_pixel_neg = np.asarray(minus_loaded[3], dtype=np.uint8)


for label, loaded in (("+", plus_loaded), ("-", minus_loaded)):
    image, mask_detector, mask_beamstop, mask_pixel_raw, metadata, exposure, source_name, dark_fit = loaded
    state = data["holo"][label]
    state["image"] = image
    state["mask_detector_raw"] = 0*mask_detector
    state["mask_beamstop_raw"] = mask_beamstop
    state["mask_pixel_raw"] = mask_pixel_raw
    stitching_masks = metadata.pop("stitching_masks", None)
    state["raw_metadata"] = metadata
    if stitching_masks is not None:
        state["stitching_masks"] = stitching_masks
    state["exposure"] = exposure
    state["spe_name"] = source_name
    state["dark_fit"] = dark_fit

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for row, label in enumerate(("+", "-")):
    state = data["holo"][label]
    vmin, vmax = wf.finite_percentile_limits(state["image"], (1, 99.9))
    axes[row, 0].imshow(state["image"], cmap="viridis", vmin=vmin, vmax=vmax)
    axes[row, 0].set_title(f"{label}: dark-corrected image")
    axes[row, 1].imshow(state["mask_detector_raw"], cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title("fixed mask_detector")
    axes[row, 2].imshow(state["mask_beamstop_raw"], cmap="gray", vmin=0, vmax=1)
    axes[row, 2].set_title("mask_beamstop")
    axes[row, 3].imshow(state["mask_pixel_raw"], cmap="gray", vmin=0, vmax=1)
    axes[row, 3].set_title("combined mask_pixel")
    for axis in axes[row]:
        axis.set_axis_off()
plt.tight_layout()
plt.show()

interactive(children=(FloatRangeSlider(value=(608399.424, 732079.0), description='contrast', layout=Layout(wid…

+: dark scale=0.998621, offset=908.028


interactive(children=(FloatRangeSlider(value=(608399.424, 732079.0), description='contrast', layout=Layout(wid…

-: dark scale=0.998725, offset=942.83


In [530]:
plt.close("all")

fig,ax=plt.subplots()
ax.imshow(pos_raw>0, cmap="gray_r")

plt.tight_layout()
plt.show()

## Choose center

In [531]:
plt.close("all")

# Starting center retained from the newer working notebook.
pol = reference_label
c0, c1 = [970, 957]
c0, c1 = [962, 894]
ic = interactive.InteractiveCenter(neg_raw, c0=c0, c1=c1)

interactive(children=(FloatRangeSlider(value=(-2463.9068526744586, 2535212.4407806224), description='contrast'…

interactive(children=(IntText(value=962, description='c0 (vert)', step=0), IntText(value=894, description='c1 …

In [532]:
# Get center positions from the widget.
center = [ic.c0, ic.c1]
print("Center:", center)
data["center"] = center
data = wf.define_centered_holograms(
    data,
    cci,
    PhR=PhR,
    project_ewalds_sphere=project_ewalds_sphere,
    ewald_method=ewald_method,
)

for label, state in data["holo"].items():
    state["mask_detector_c"] = (
        wf.center_image(state["mask_detector_raw"], data["center"], cci) > 0.5
    ).astype(np.uint8)
    state["mask_beamstop_c"] = (
        wf.center_image(state["mask_beamstop_raw"], data["center"], cci) > 0.5
    ).astype(np.uint8)
    # Transform the authoritative combined mask, including stitching holes.
    state["mask_pixel_c"] = (
        wf.center_image(state["mask_pixel_raw"].astype(float), data["center"], cci) > 0.5
    ).astype(np.uint8)
# A difference hologram is valid only where both input images are valid.
mask_pixel = np.maximum(
    data["holo"][positive_label]["mask_pixel_c"],
    data["holo"][reference_label]["mask_pixel_c"],
).astype(np.uint8)
data["mask_pixel_raw_by_label"] = {
    label: state["mask_pixel_raw"] for label, state in data["holo"].items()
}
data["mask_pixel"] = mask_pixel
plot_labels = list(data["holo"].keys())
print("Plot order:", plot_labels)
cimshow(np.stack([data["holo"][label]["image_c"] for label in plot_labels]))

Center: [962, 894]
Plot order: ['+', '-']


interactive(children=(FloatRangeSlider(value=(-2439.669231118046, 2547333.019560393), description='contrast', …

interactive(children=(IntSlider(value=0, description='nr', max=1), Output()), _dom_classes=('widget-interact',…

(<Figure size 700x700 with 1 Axes>, <Axes: >)

## Smooth Butterworth mask and FTH reconstruction

In [533]:
# Optional horizontal-band correction. Parameters stay beside the operation.
CORRECT_HORIZONTAL_BAND = False
BAND_EDGE_COLUMNS = 20
BAND_SKIPPED_EDGE_COLUMNS = 0
BAND_POLYNOMIAL_ORDER = 2
BAND_CENTER = 1024
BAND_WIDTH = 80
BAND_EDGE = 11
SUBTRACT_POLYNOMIAL_BACKGROUND = False


In [534]:
# FTH masking parameters are here because the masks are constructed below.
# mask_pixel remains binary for phase retrieval; these smooth masks are FTH-only.
MASK_PIXEL_DILATION = 3
MASK_PIXEL_BLUR_SIGMA = 3.0
BUTTERWORTH_RADIUS = 35
BUTTERWORTH_ORDER = 4
prop_dist = 0
phase = 0

shape = data["holo"][positive_label]["image_c"].shape
if mask_pixel.shape != shape:
    raise ValueError(f"Centered mask shape {mask_pixel.shape} != hologram shape {shape}")

# 1) Slightly enlarge mask_pixel, then blur its edge.
mask_pixel_fth = wf.smooth_binary_mask(
    mask_pixel.astype(float),
    MASK_PIXEL_DILATION,
    MASK_PIXEL_BLUR_SIGMA,
)

# 2) Apply a smooth Butterworth stop around the Fourier origin.
mask_beamstop_smooth = wf.butterworth_disk_mask(
    shape,
    BUTTERWORTH_RADIUS,
    BUTTERWORTH_ORDER,
)

# Both arrays above are exclusion masks: 0 is retained and 1 is removed.
mask_multiplier = (1 - mask_pixel_fth) * (1 - mask_beamstop_smooth)

mask_pixel_fth_recipe = {
    "type": "binary_dilation_gaussian",
    "dilation_pixels": MASK_PIXEL_DILATION,
    "sigma": MASK_PIXEL_BLUR_SIGMA,
}
mask_beamstop_smooth_recipe = {
    "type": "radial_butterworth_disk",
    "radius": BUTTERWORTH_RADIUS,
    "order": BUTTERWORTH_ORDER,
}

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for axis, shown_mask, title in zip(
    axes,
    (mask_pixel, mask_pixel_fth, mask_beamstop_smooth, mask_multiplier),
    ("binary mask_pixel", "dilated + blurred mask_pixel",
     "Butterworth exclusion mask", "final FTH transmission"),
):
    image = axis.imshow(shown_mask, cmap="gray", vmin=0, vmax=1)
    axis.set_title(title)
    axis.set_axis_off()
    fig.colorbar(image, ax=axis, fraction=0.046)
plt.tight_layout()
plt.show()

pos = np.asarray(data["holo"][positive_label]["image_c"], dtype=float)
neg = np.asarray(data["holo"][reference_label]["image_c"], dtype=float)
if CORRECT_HORIZONTAL_BAND:
    band_fits = []
    for label, image in ((positive_label, pos), (reference_label, neg)):
        band_fit = fit_horizontal_band(
            image,
            edge_columns=BAND_EDGE_COLUMNS,
            skipped_edge_columns=BAND_SKIPPED_EDGE_COLUMNS,
            polynomial_order=BAND_POLYNOMIAL_ORDER,
            band_center=BAND_CENTER,
            band_width=BAND_WIDTH,
            band_edge=BAND_EDGE,
            mask=mask_pixel,
        )
        band_fits.append(band_fit)
        fig, axis = plt.subplots(figsize=(10, 4))
        axis.plot(band_fit.rows, band_fit.measured_profile, label="measured edge profile")
        axis.plot(band_fit.rows, band_fit.fitted_profile, linewidth=2, label="complete fit")
        axis.plot(band_fit.rows, band_fit.polynomial_profile, "--", label=f"polynomial order {BAND_POLYNOMIAL_ORDER}")
        axis.fill_between(band_fit.rows, band_fit.polynomial_profile, band_fit.fitted_profile, alpha=0.25, label="negative band")
        axis.set_title(f"{label}: horizontal-band fit")
        axis.set_xlabel("Detector row [pixel]")
        axis.set_ylabel("Intensity")
        axis.legend()
        axis.grid(alpha=0.2)
        plt.tight_layout()
        plt.show()
    pos = pos - band_fits[0].band_image
    neg = neg - band_fits[1].band_image
    if SUBTRACT_POLYNOMIAL_BACKGROUND:
        pos = pos - band_fits[0].polynomial_image
        neg = neg - band_fits[1].polynomial_image

# Fit polarization normalization only outside mask_detector + mask_beamstop.
POLARIZATION_FIT_ROWS = slice(500, -500)
POLARIZATION_FIT_COLUMNS = slice(500, -500)
POLARIZATION_FIT_PERCENTILES = (2, 98)

fit_region = np.zeros(pos.shape, dtype=bool)
fit_region[POLARIZATION_FIT_ROWS, POLARIZATION_FIT_COLUMNS] = True
fit_pixels = (
    (mask_pixel == 0)
    #& fit_region
    #& np.isfinite(pos)
    #& np.isfinite(neg)
)
low, high = np.percentile(pos[fit_pixels], POLARIZATION_FIT_PERCENTILES)
#fit_pixels &= (pos >= low) & (pos <= high)
factor, offset = cci.dyn_factor(
    pos[fit_pixels],
    neg[fit_pixels],
    method="scalarproduct",
    verbose=False,
    plot=False,
)
pos_normalized = pos / factor

plot_step = max(1, fit_pixels.sum() // 7000)
fit_x_values = pos[fit_pixels][::plot_step]
fit_y_values = neg[fit_pixels][::plot_step]
fit_x = np.linspace(fit_x_values.min(), fit_x_values.max(), 300)
fig, axis = plt.subplots(figsize=(6, 5))
axis.scatter(fit_x_values, fit_y_values, s=3, alpha=0.15, label="valid fit pixels")
axis.plot(fit_x, fit_x / factor - offset, color="red", linewidth=2,
          label=f"fit: x/{factor:.5g} - {offset:.5g}")
axis.set_title("Polarization normalization outside mask_pixel")
axis.set_xlabel(f"{positive_label} intensity")
axis.set_ylabel(f"{reference_label} intensity")
axis.legend()
axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()
holo_unmasked = pos_normalized - neg - offset
holo_masked = holo_unmasked * mask_multiplier
recon_unmasked = wf.fth_reconstruct(
    holo_unmasked, experimental_setup, fth, prop_dist=prop_dist, phase=phase
)
recon_masked = wf.fth_reconstruct(
    holo_masked, experimental_setup, fth, prop_dist=prop_dist, phase=phase
)

data["mask_pixel"] = mask_pixel
data["mask_beamstop_smooth_recipe"] = mask_beamstop_smooth_recipe
data["mask_pixel_fth_recipe"] = mask_pixel_fth_recipe
data["factor"] = factor
data["offset"] = offset
data["holo"][positive_label]["image_c_norm"] = wf.normalize_image(pos_normalized)
data["holo"][reference_label]["image_c_norm"] = wf.normalize_image(neg)
data["focus_fth"] = {
    "prop_dist": prop_dist,
    "prop_dist_unit": "um",
    "phase": phase,
    "dx": 0,
    "dy": 0,
    "roi": None,
    "operation": "-",
}
data["fth_recipe"] = {
    "positive_label": positive_label,
    "reference_label": reference_label,
    "center": data["center"],
    "mask_beamstop_smooth_recipe": mask_beamstop_smooth_recipe,
    "mask_pixel_fth_recipe": mask_pixel_fth_recipe,
    "project_ewalds_sphere": data["project_ewalds_sphere"],
    "ewald_method": ewald_method,
    "contrast": "positive / factor - reference - offset",
}

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
tmp = holo_masked
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (0.1, 99.9))
ax[0].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[0].add_patch(
    plt.Circle(
        (shape[1] / 2, shape[0] / 2),
        BUTTERWORTH_RADIUS,
        fill=False,
        edgecolor="red",
        linewidth=1.5,
    )
)
ax[0].set_title("mask * hologram")

tmp = np.real(recon_unmasked)
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax[1].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[1].set_title("FTH before masking")
tmp = np.real(recon_masked)
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax[2].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[2].set_title("FTH after masking")
for axis in ax:
    axis.set_axis_off()
plt.show()

In [535]:
fig, ax = plt.subplots()
ax.imshow(pos, vmin=0, vmax=100)

## Define ROI from masked FTH reconstruction

In [536]:
plt.close("all")
holo_unmasked = pos_normalized + neg 
holo_masked = holo_unmasked * mask_multiplier
recon_unmasked = wf.fth_reconstruct(
    holo_unmasked, experimental_setup, fth, prop_dist=prop_dist, phase=phase
)
recon_masked = wf.fth_reconstruct(
    holo_masked, experimental_setup, fth, prop_dist=prop_dist, phase=phase
)

# Preview the masked FTH reconstruction. The next cell applies the calibrated fixed ROI.
fig, ax = cimshow(np.real(recon_masked))

interactive(children=(FloatRangeSlider(value=(-4792759132.492979, 4026356115.9992976), description='contrast',…

In [537]:
# Fixed ROI calibrated with the image-13 center above.
roi = [597, 785, 526, 719]
roi = [300,800, 1250, 1750]
roi_s = np.s_[roi[0] : roi[1], roi[2] : roi[3]]
print("ROI:", roi)

data["roi"]=roi

ROI: [300, 800, 1250, 1750]


In [538]:
fig,ax=plt.subplots()

import mask_lib
mask11=1-mask_lib.circle_mask(mask_pixel.shape,
                              center=[mask_pixel.shape[0]//2,mask_pixel.shape[0]//2],
                            radius=100,sigma=5)
ax.imshow(mask_pixel, cmap="Blues")
ax.imshow(mask11, alpha=0.5, cmap="Reds")

## Focus FTH reconstruction

In [539]:
# Preview is available after temp_p and temp_n are defined below.
im_ids

[1868]

In [540]:
plt.close("all")
import fth_phase_workflow as wf
# Use the focusCDI widget to tune propagation distance and phase.
# The selected values are committed to the HDF5 data dictionary in the next cell.
# Manual starting values: propagation is in micrometres, phase in radians,
# and dx/dy are sub-pixel shifts. Edit these before creating the sliders.



prop_dist, phase, dx, dy = 7.36,-0.3-np.pi/2,0,0
prop_dist, phase, dx, dy = 9.35,1.59,0,0
prop_dist, phase, dx, dy = 23.98,-1.15,-0.24,0
prop_dist, phase, dx, dy = 28.55,-0.56,-0.24,0
#prop_dist, phase, dx, dy = 26.75,0.20,-0.12,-0.09
# Use the same normalization and transmission as the saved FTH recipe.
temp_p = pos_normalized * mask_multiplier#*mask11
temp_n = (neg + offset) * mask_multiplier#*mask11

focus_operation = "-"
focus_sliders = rec.focusCDI(
    temp_p,temp_n,
    roi_s,
    mask=1,
    phase=phase,
    prop_dist=prop_dist,
    dx=dx,
    dy=dy,
    experimental_setup=data["experimental_setup"],
    operation=focus_operation,
    max_prop_dist=85,
    #scale=(0.1, 99.9),
    scale=(0, 100),
)
slider_prop, slider_phase, slider_dx, slider_dy = focus_sliders[:4]

interactive(children=(FloatSlider(value=28.55, description='propagation[um]', layout=Layout(width='90%'), max=…

In [288]:
RUN_OPTIONAL_FOCUS_EXPERIMENTS = False


In [289]:
if RUN_OPTIONAL_FOCUS_EXPERIMENTS:
    reload(rec)
    roi_s2=np.s_[750:825,1200:1300]
    sliders = rec.focusCDIdeconv(
        temp_p,temp_n,
        roi_s2,
        mask=1,
        phase=-1.87,
        prop_dist=7.36,
        dx=0,
        dy=0,
        operation="-",
        reference_profile="Gaussian",
        reference_diameter=5.7,  # Reference hole diameter in nm
        regularization=1e-3,
        deconvolution=True,
        max_prop_dist=20,
        experimental_setup=experimental_setup
    )


In [290]:
if RUN_OPTIONAL_FOCUS_EXPERIMENTS:
    reload(rec)
    
    controls = rec.focusCDI_complex(
        temp_p,temp_n,
        roi_s2,
        mask=1,
        operation="-",
        phase=-1.87,
        prop_dist=7.36,
        dx=0,
        dy=0,
        reference_diameter=5,
        reference_profile="Circular",
        regularization=1e-3,
        deconvolution=True,
        experimental_setup=experimental_setup
    )


In [291]:
if RUN_OPTIONAL_FOCUS_EXPERIMENTS:
    reload(rec)
    result = rec.optimize_focusCDI(
        temp_p,temp_n,
        roi_s2,
        experimental_setup=experimental_setup,
        operation="-",
        profiles=("Circular", "Gaussian", "Lorentzian"),
        prop_bounds=(0, 13),
        diameter_bounds=(1.5, 8),
        maxiter=30,
        component="real"
    )
    
    print(result["parameters"])


In [292]:
if RUN_OPTIONAL_FOCUS_EXPERIMENTS:
    fig, axs = plt.subplots(1, 3, figsize=(12, 4))
    
    image = result["image"]
    
    axs[0].imshow(np.abs(image), cmap="gray")
    axs[0].set_title("Amplitude")
    
    axs[1].imshow(np.real(image), cmap="gray")
    axs[1].set_title("Real")
    
    axs[2].imshow(np.imag(image), cmap="gray")
    axs[2].set_title("Imaginary")
    
    plt.tight_layout()


In [293]:
if RUN_OPTIONAL_FOCUS_EXPERIMENTS:
    plt.close("all")
    
    crop=200
    roicrop=np.s_[crop:-crop,crop:-crop]
    
    focus_sliders = rec.focusCDI(
        temp_p[roicrop],temp_n[roicrop],
        np.s_[920:1080,550:720],
        mask=1,
        phase=phase,
        prop_dist=prop_dist,
        dx=dx,
        dy=dy,
        experimental_setup=data["experimental_setup"],
        operation="-",
        max_prop_dist=35,
        #scale=(0.1, 99.9),
        scale=(0, 100),
    )


In [541]:
def p(pos, neg,factor, x, y, fx, fy, operation,roi):
    image_p = fth.reconstructCDI(fth.propagate(pos, x*1e-6, experimental_setup)* np.exp(1j*y))
    image_n = fth.reconstructCDI(fth.propagate(neg, x*1e-6, experimental_setup)* np.exp(1j*y))
    
    if operation== "-":
        image= (image_p-image_n)
    elif operation== "+":
        image= (image_p+image_n)
    elif operation=="/":
        image= np.log((image_p/image_n))* np.exp(1j*y)
    elif operation=="**2/**2":
        image= (np.abs(image_p)**2/np.abs(image_n)**2) * np.exp(1j*y)
    elif operation=="-/+":
        image= (image_p-image_n)/(image_p+image_n) * np.exp(1j*y)
    elif operation=="load_both":
        image,_=fth.load_both(pos,neg,True)
        image= fth.reconstructCDI(fth.propagate(image, x*1e-6, experimental_setup))* np.exp(1j*y)
        
    image=np.nan_to_num(image, nan=0, posinf=0, neginf=0)
    simage = fth.sub_pixel_centering(image, fx, fy)[roi]

    return simage


recon=p(temp_p,temp_n,factor,slider_prop.value,
        slider_phase.value,slider_dx.value,
        slider_dy.value, focus_operation,roi_s)

data["recon"]=recon.copy()

data["focus_fth"] = {
    "prop_dist": float(slider_prop.value),
    "prop_dist_unit": "um",
    "phase": float(slider_phase.value),
    "dx": float(slider_dx.value),
    "dy": float(slider_dy.value),
    "roi": roi,
    "operation": focus_operation,
}

In [542]:
fig, ax = plt.subplots(figsize=(5, 5))
tmp = np.real(data["recon"])
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (0.1, 99.9))
ax.imshow(tmp,cmap="gray") #,vmin=vmin, vmax=vmax) 

## Save

In [543]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(tmp,cmap="gray")

In [544]:
# Always write the display PNG in the same final cell as the HDF5 result.
im_ids = image_ids(data["holo"][positive_label]["id"])
im_id = first_id(im_ids)
png_name = join(folder_general, f"FTH_recon_ImId_{int(im_id):04d}_{USER}.png")
fig, ax = plt.subplots(figsize=(5, 5))
tmp = np.real(data["recon"])
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (0.5, 99))
ax.imshow(tmp, cmap="gray")#,vmin=vmin, vmax=vmax)
ax.set_title(f"Focused FTH after masking - im_id {im_id}")
ax.set_axis_off()
fig.savefig(png_name, bbox_inches="tight", transparent=False, dpi=200)
plt.close(fig)
data["fth_png"] = png_name
written = wf.save_data_dict(data, DATA_H5, overwrite=True)
print("Saved HDF5:", written)
print("Saved PNG:", png_name)

Saved HDF5: /home/experiences/sextants/com-sextants/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS-2609_SOLEIL_SEXTANTS/processed/Logs/data_recon_ImId_1868_rb.hdf5
Saved PNG: /home/experiences/sextants/com-sextants/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS-2609_SOLEIL_SEXTANTS/processed/FTH_recon_ImId_1868_rb.png


In [545]:
# Workflow summary
_summary_data = data if "data" in globals() and isinstance(data, dict) else {}
_summary_h5 = globals().get("DATA_H5", _summary_data.get("data_file", "n/a"))
_summary_holo = _summary_data.get("holo", {})
_summary_pos = _summary_data.get(
    "positive_label", globals().get("positive_label", None)
)
_summary_ref = _summary_data.get(
    "reference_label", globals().get("reference_label", None)
)
_summary_im = _summary_holo.get(_summary_pos, {}).get(
    "id", globals().get("im_id", "n/a")
)
_summary_topo = _summary_holo.get(_summary_ref, {}).get(
    "id", globals().get("topo_id", "n/a")
)
print("im_ids:", _summary_im)
print("topo_ids:", _summary_topo)
print("dark_ids (+):", _summary_holo.get(_summary_pos, {}).get("dark_id"))
print("dark_ids (-):", _summary_holo.get(_summary_ref, {}).get("dark_id"))
print("HDF5:", _summary_h5)

im_ids: [1868]
topo_ids: [1869]
dark_ids (+): 1837
dark_ids (-): 1837
HDF5: /home/experiences/sextants/com-sextants/FTH-Phase-Retrieval-2609-SOLEIL-SEXTANTS-2609_SOLEIL_SEXTANTS/processed/Logs/data_recon_ImId_1868_rb.hdf5


In [55]:
# Acquisition ID summary
_id_holo = data.get("holo", {})
print("im_ids:", {label: state.get("id") for label, state in _id_holo.items()})
print("dark_ids:", {label: state.get("dark_id") for label, state in _id_holo.items()})

im_ids: {'+': [1564], '-': [1565]}
dark_ids: {'+': 904, '-': 904}


## Simple names for interactive inspection

Use these names directly when making an extra plot. Holograms use `viridis`; `reconstruction_fth` is a real-space image and may be shown in gray.


In [23]:
# Conventional short NumPy names used throughout older FTH notebooks.
pos = np.asarray(pos, dtype=float)
neg = np.asarray(neg, dtype=float)
holo = np.asarray(holo_masked, dtype=float)
recon = np.asarray(data["recon"])


def show_image(image, title="image", cmap="viridis", percentiles=(1, 99.9)):
    image = np.asarray(image)
    values = np.real(image) if np.iscomplexobj(image) else image.astype(float)
    finite = values[np.isfinite(values)]
    vmin, vmax = np.percentile(finite, percentiles)
    fig, axis = plt.subplots(figsize=(6, 5))
    shown = axis.imshow(values, cmap=cmap, vmin=vmin, vmax=vmax)
    axis.set_title(title)
    axis.set_axis_off()
    fig.colorbar(shown, ax=axis)
    plt.tight_layout()
    plt.show()
    return fig, axis


print("Easy NumPy names: pos_raw, neg_raw, pos, neg, mask_pixel, holo, recon")
print("Example: show_image(pos, 'positive image')")
print("Example: show_image(holo, 'FTH hologram')")
print("Example: show_image(recon, 'FTH reconstruction', cmap='gray')")


Easy NumPy names: pos_raw, neg_raw, pos, neg, mask_pixel, holo, recon
Example: show_image(pos, 'positive image')
Example: show_image(holo, 'FTH hologram')
Example: show_image(recon, 'FTH reconstruction', cmap='gray')
